# Decoder 训练 + 闭环 rollout 评估（CartPole + Pendulum）

这个 notebook 包含两个并列的部分：

1. **Decoder 训练**：对应 `train_decoder.py`，config 在 `config/train_decoder/<环境>/<weight_mode>.json`，
   三组 loss 对比 + 单帧指标汇总。
2. **闭环 rollout 评估**：对应 `sampling.py`，生成 DWM 轨迹并和 real 轨迹做逐步偏差对比，
   是独立于单帧指标的决定性评估（单帧好 ≠ rollout 好，hybrid 是历史反例）。

三组训练实验唯一区别 = loss 里权重 w 的来源（详见 `report/2026-07-06.md` 第 4 节）：
- `intensity` -> 论文 baseline（暗像素规则，Eq.7）
- `saliency`  -> 我们的方法（w = 1 + α·H，H 为 occlusion 热图）
- `hybrid`    -> 诊断组（已否决：intensity 会稀释 saliency）

MountainCar 当前冻结（controller 有问题，occlusion heatmap 关注区域异常），
`config/train_decoder/mountain_car/` 是空预留位，修好 controller 之前不训。

**运行前**：kernel 切换成 `starv_shared`（`/home/tealab_shared/starv/env/starv_shared`）。
本文件放在 `verifiable_wm/notebooks/` 下，第一个 cell 会自动定位仓库根目录。

In [2]:
import os
os.environ.setdefault("PYGLET_HEADLESS", "True")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dynamic.py").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "dynamic.py").exists(), (
    f"没找到仓库根目录（dynamic.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

repo root: /home/UFAD/cjiang2/VWM/verifiable_wm


In [3]:
from utils import load_config, set_seed, resolve_device
import train_decoder as td

def run_train(env_name, weight_mode, alpha=None, seed=None, output_dir=None):
    """跑一组 decoder 训练。
    weight_mode: intensity / saliency / hybrid
    alpha / seed: 不填用 config 默认值；填了则覆盖，且输出目录自动加后缀避免覆盖主结果
    """
    config_path = REPO_ROOT / "config" / "train_decoder" / env_name / f"{weight_mode}.json"
    config = load_config(config_path)
    if alpha is not None:
        config["weight"]["alpha"] = alpha
    if seed is not None:
        config["training"]["seed"] = seed
    if output_dir is not None:
        config["output_dir"] = str(output_dir)
    else:
        suffix = ""
        if alpha is not None:
            suffix += f"_alpha{alpha:g}"
        if seed is not None:
            suffix += f"_seed{seed}"
        config["output_dir"] += suffix
    print(f"[Config] {config_path}")
    print(f"[Output] {config['output_dir']}")
    set_seed(config["training"]["seed"])
    device = resolve_device(config.get("device", "auto"))
    td.train(config, device)

# 第一部分：Decoder 训练

## Case 1: CartPole - intensity（论文 baseline）

In [ ]:
run_train("cartpole", "intensity")

## Case 2: CartPole - saliency（我们的方法）

In [ ]:
run_train("cartpole", "saliency")

## Case 3: CartPole - hybrid（诊断组，已否决，一般不用重跑）

In [ ]:
# run_train("cartpole", "hybrid")

## α 扫描（下一步计划）

在 val controller MSE 上选最优 α；α=4 的结果已在主目录，无需重跑。

In [ ]:
# for alpha in [1, 2, 8, 16]:
#     run_train("cartpole", "saliency", alpha=alpha)

## Case 4: Pendulum - intensity（论文 baseline）

In [ ]:
run_train("pendulum", "intensity")

## Case 5: Pendulum - saliency（我们的方法）

In [ ]:
run_train("pendulum", "saliency")

## 汇总：读取各实验的 test 指标

In [4]:
import json
from pathlib import Path

def print_metrics_summary(env_name):
    print(f"[{env_name}]")
    print(f"{'实验':<24}{'best@':>6}{'ctrl_mse':>12}{'pixel_mse':>12}")
    for mfile in sorted(Path(f"dwm_weight/now_weight/{env_name}").glob("*/metrics.json")):
        m = json.load(open(mfile))
        t = m["test"]
        print(f"{mfile.parent.name:<24}{m['best_epoch']:>6}{t['ctrl_mse']:>12.5f}{t['pixel_mse']:>12.5f}")

print_metrics_summary("cartpole")
print_metrics_summary("pendulum")

[cartpole]
实验                       best@    ctrl_mse   pixel_mse
hybrid                       8     0.01030     0.01678
intensity                  199     0.01896     0.00367
saliency                   199     0.00052     0.00188
[pendulum]
实验                       best@    ctrl_mse   pixel_mse
intensity                  199     0.00067     0.00167
saliency                   199     0.00041     0.00085


# 第二部分：闭环 rollout 评估

## 生成 WM 轨迹（每个变体约 2 分钟，已生成过可跳过）

用 decoder 跑 30 步闭环（`sampling.py` 的 `rollout_dwm_trajectory`），
与真实轨迹（`real_trajectories.npz`，`make_decoder_dataset.py` 的 `rollout_real_trajectory` 生成）
同起点逐步对比。等价命令行：`python sampling.py config/sampling/<env>.json --decoder-variant saliency`

In [ ]:
import sampling as sp

def run_sampling_variant(env_name, variant):
    config = load_config(REPO_ROOT / "config" / "sampling" / f"{env_name}.json")
    config["decoder"]["variant"] = variant
    sp.generate_dataset(config)

# run_sampling_variant("cartpole", "intensity")
# run_sampling_variant("cartpole", "saliency")
# run_sampling_variant("pendulum", "intensity")
# run_sampling_variant("pendulum", "saliency")

## 轨迹偏差对比（决定性指标②）

每对轨迹逐步算全状态 L2 偏差 d_t = ||s_t_wm − s_t_real||；
「轨迹最大偏差」即论文 conformal inflation（Eq.23）用的量，p95 直接预演 Δ_0.95 的松紧。

In [5]:
import numpy as np

def wrapped_diff(a, b, period):
    """Shortest signed difference between two points on a circle of `period`.
    Needed for state dims that wrap (e.g. pendulum theta, wrapped to
    [-pi, pi] every step by dynamic.py's Pendulum.step). Plain subtraction
    of two already-wrapped angles can spuriously read ~period even when the
    two angles are right next to each other on the circle (one just before
    the seam, one just after)."""
    d = a - b
    half = period / 2.0
    d = np.where(d > half, d - period, d)
    d = np.where(d < -half, d + period, d)
    return d

def print_rollout_deviation(env_name, variants, circular_dims=(), period=2 * np.pi, ord=2):
    """ord=2 (L2) matches what this cell has always used; the paper's Theorem 1
    non-conformity score (Eq. 1 / Eq. 23) is defined with ord=1 (L1) -- switch
    to ord=1 if/when these numbers need to feed the conformal-inflation step
    rather than just compare decoder variants descriptively."""
    root = Path(f"datasets/{env_name}/data/dataset_v1")
    real = np.load(root / "real_trajectories.npz")

    print(f"[{env_name}] circular_dims={list(circular_dims)} ord={ord}")
    print(f"{'变体':<12}{'平均每步':>10}{'末步':>10}{'最大偏差均值':>14}{'最大偏差p95':>13}")
    for variant in variants:
        f = root / f"dwm_trajectories_{variant}.npz"
        if not f.exists():
            print(f"{variant:<12} (未生成)")
            continue
        dwm = np.load(f)
        r, w = real["test_traj"], dwm["test_traj"]
        assert (r[:, 0] == w[:, 0]).all(), "起点不一致，无法配对对比"

        diff = w - r
        if circular_dims:
            diff = diff.copy()
            for dim in circular_dims:
                diff[..., dim] = wrapped_diff(w[..., dim], r[..., dim], period)
        d = np.linalg.norm(diff, ord=ord, axis=2)   # (N, 31) 每步偏差
        dmax = d.max(axis=1)                        # 每条轨迹的最大偏差
        print(f"{variant:<12}{d.mean():>10.4f}{d[:, -1].mean():>10.4f}"
              f"{dmax.mean():>14.4f}{np.percentile(dmax, 95):>13.4f}")

print_rollout_deviation("cartpole", ["old", "intensity", "saliency"])
print_rollout_deviation("pendulum", ["old", "intensity", "saliency"], circular_dims=[0])

[cartpole] circular_dims=[] ord=2
变体                平均每步        末步        最大偏差均值      最大偏差p95
old             0.5369    1.2071        1.2109       5.2191
intensity       0.5014    1.0750        1.0788       4.6685
saliency        0.0736    0.1809        0.1834       0.5942
[pendulum] circular_dims=[0] ord=2
变体                平均每步        末步        最大偏差均值      最大偏差p95
old             0.1345    0.3500        0.4048       0.9941
intensity       0.0581    0.1515        0.1806       0.5675
saliency        0.0563    0.1556        0.1822       0.5289
